# PSF, sources and realism (euclid-like)

This notebook explores more realistic PSF wavelength dependence, source types
(stars, galaxies, planetary nebulae), and photometry choices for euclid-like
simulations. Use it as a playground to understand how astrophysical source
morphologies and instrument PSF shape the observed dispersed images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spectangle.physics.psf import PSFModel
from spectangle.simulations.sed import BlackbodySED, RealisticSED

# wavelength-dependent FWHM: assume diffraction-limited plus small aberration
wavs = np.array([9250, 11000, 13500, 16000, 18500], dtype=float)
fwhm_ref = 1.6
fwhms = fwhm_ref * (wavs / 13500.0) + 0.2  # add small floor

kernels = [PSFModel(fwhm_pixels=f).kernel for f in fwhms]
labels = [f"{int(w)} Å - FWHM={f:.2f}px" for w,f in zip(wavs, fwhms)]

# display
fig = plt.figure(figsize=(12, 4))
for i, k in enumerate(kernels):
    ax = fig.add_subplot(1, len(kernels), i+1)
    ax.imshow(k, origin='lower', cmap='viridis')
    ax.set_title(labels[i])
    ax.axis('off')
plt.suptitle('Euclid-like PSF kernels (toy scaling)')
plt.tight_layout()
plt.show()

## Source types
We will synthesise different astrophysical sources:
- Stars: point sources with blackbody or realistic stellar spectra.
- Galaxies: exponential/Sérsic approximations with spatial extent and continuum SED.
- Planetary nebulae: emission-line dominated, ring-like morphology.

We'll create a small library of stamps for each type that can be used in
forward simulations.

In [ ]:
from scipy.ndimage import gaussian_filter

ny, nx = 128, 128
n_lambda = 256
wav = np.linspace(9250, 18500, n_lambda)

# Stars
star_temp = 5800
star_sed = BlackbodySED(star_temp)
star_stamp = np.zeros((n_lambda, ny, nx), dtype=np.float32)
cy, cx = ny//2, nx//2
for i, lam in enumerate(wav):
    star_stamp[i, cy, cx] = star_sed(np.array([lam]))[0]

# Galaxy: exponential disk approx
sigma = 6.0
xv, yv = np.meshgrid(np.arange(nx), np.arange(ny))
rg = np.sqrt((xv - nx*0.6)**2 + (yv - ny*0.45)**2)
gal_spatial = np.exp(-rg/ sigma)
gal_stamp = np.stack([gal_spatial * BlackbodySED(4500)(np.array([lam]))[0] for lam in wav], axis=0)

# PN: ring with narrow emission line
pn_stamp = np.zeros((n_lambda, ny, nx), dtype=np.float32)
line_center = 13700
for i, lam in enumerate(wav):
    if abs(lam - line_center) < 80:
        r = np.sqrt((xv - nx*0.3)**2 + (yv - ny*0.65)**2)
        ann = np.logical_and(r > 8, r < 12).astype(float)
        pn_stamp[i] = ann / ann.sum()

# Quick visualisation: broadband
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].imshow(star_stamp.sum(axis=0), origin='lower', cmap='magma')
axs[0].set_title('Star (broadband)')
axs[1].imshow(gal_stamp.sum(axis=0), origin='lower', cmap='magma')
axs[1].set_title('Galaxy (broadband)')
axs[2].imshow(pn_stamp.sum(axis=0), origin='lower', cmap='magma')
axs[2].set_title('Planetary Nebula (broadband)')
for ax in axs:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Using these stamps in forward simulations
You can place multiple stamps at arbitrary positions in a scene cube and then
use the ForwardModel to produce synthetic K-sequence spectrograms. Try
varying PSF, exposure time, and source luminosity to explore detection and
confusion regimes.

In [ ]:
from spectangle.simulations.forward import ForwardModel
from spectangle.physics.dispersion import KSequence
from spectangle.physics.psf import PSFModel

# Build scene with placed sources
scene = np.zeros((n_lambda, ny, nx), dtype=np.float32)
scene[:, 40:40+star_stamp.shape[1], 20:20+star_stamp.shape[2]] += star_stamp
scene[:, 70:70+gal_stamp.shape[1], 60:60+gal_stamp.shape[2]] += gal_stamp
scene[:, 20:20+pn_stamp.shape[1], 80:80+pn_stamp.shape[2]] += pn_stamp

kseq = KSequence.miniature(n_lambda)
psf = PSFModel(fwhm_pixels=1.8)
fwd = ForwardModel(ksequence=kseq, psf_model=psf, image_shape=(ny, nx), orders=[1])

specs = fwd(scene, wav)

# plot spectrograms
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for k in range(4):
    im = ax[k].imshow(specs[k], origin='lower', cmap='inferno')
    ax[k].axis('off')
    ax[k].set_title(f'K-step {k+1}')
    fig.colorbar(im, ax=ax[k], fraction=0.04)
plt.tight_layout()
plt.show()

Next steps: use the complex simulator to add noise and multi-order effects, and
then try training the PINN to recover line emitters (PN) and extended sources
(galaxies) — these are common failure modes for simple reconstruction nets.